# 23. XGB에 절단 변이(trunc) 피처 넣기 (H23)

## 배경 (11/11 회의)
- `trunc` = 유전자마다 절단 변이(넌센스·프레임시프트)가 1개 이상 있으면 1 (`유전자__trunc`)
  - 프레임시프트의 **위치는 보지 않음** → 핫스팟(11, LB 실패)처럼 정확한 위치를 외우는 피처가 아님
- 지금까지 trunc는 **LR에만** 넣어 봄 (07 F, 16의 LR_trunc), XGB에는 넣은 적 없음
- 20에서 정규화 후 test에서도 trunc가 같은 방식으로 계산되는 것 확인 (변이 칸 중 절단 변이 칸 train 10.2% / test 10.5%)

## 회의에서 미뤘던 이유 (이번에 확인할 위험)
1. LR_trunc 비중이 가장 컸던 select6(0.38)이 LB 0.3183으로 14(0.3247)보다 낮았음
2. test에서 APC 절단 변이 환자가 3.5% → 24.2%로 크게 늘었고, train에서 APC 절단 변이 환자의 71%가 COAD
   → XGB가 `APC__trunc`를 직접 보면 test COAD 예측이 더 늘 수 있음

## 가설
- H23a: 기준(any + stats)에 trunc를 더하면 CV가 오른다
- H23b: 지금 최고 피처인 22b(any + ratio + 치환 비율 상위 100)에 trunc를 더하면 22b보다 오른다
- 위험 확인 (관찰용): trunc를 넣으면 test COAD 예측 비율과 APC 절단 변이 환자의 COAD 예측이 늘어나는지

## 검증 방법
- 모델: 10 최적 설정 XGB 고정 (depth 3, lr 0.1, 200그루, colsample 0.3), 데이터: `load_data()` (정규화)
- 비교 (같은 fold, 피처만 다름): 기준 vs 23a, 22b vs 23b
- trunc 열은 유전자 수만큼 많고 대부분 0 → **fold 학습 부분에서 한 번도 1이 아닌 trunc 열은 뺌** (colsample이 빈 열을 뽑지 않게, fold 안에서만 결정 → 누수 아님)
- 22b 피처(`ratio`, 치환 비율)는 22 노트북과 같은 계산을 이 노트북 안에 넣음 → 22 PR 머지 여부와 상관없이 실행 가능
- 채택 기준: 비교 대상보다 5 fold 중 4개 이상 상승
- 실행 환경: macOS M1 (XGB 점수는 데스크탑과 ±0.004 차이 가능), 제출 파일은 Windows에서만 저장

#### 준비: 데이터, 기본 피처, 22b 피처, 10 최적 설정

In [1]:
import sys; sys.path.append('..')
import time, platform, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, mutation_type, aa_substitution, MUT_TYPES
from src.validation import seed_everything, encode_target, get_folds, run_cv, save_submission

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
folds = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

# 22a ratio (22 PR의 mutation_features()['ratio']와 같은 계산): 유형별 비율(in-frame indel 제외) + 전체 변이 수 log
def ratio_feats(df):
    keep = [t for t in MUT_TYPES if t != 'inframe_indel']
    rows = []
    for row in df[genes].to_numpy():
        c = Counter(mutation_type(t) for v in row if v != WT for t in v.split())
        tot = sum(c[t] for t in keep)
        r = {f'r_{t}': c[t] / max(tot, 1) for t in keep}
        r['n_mut_log'] = np.log1p(tot)
        rows.append(r)
    return pd.DataFrame(rows, index=df.index).astype(np.float32)

# 22b: 미스센스 치환 종류 비율, 상위 100개는 fold 학습 부분에서만
def sub_ratio_matrix(df):
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

class FoldSelect(BaseEstimator, TransformerMixin):
    # fold 학습 부분에서만 결정: 치환 비율 상위 n개만 남기기 + 한 번도 1이 아닌 trunc 열 빼기
    def __init__(self, n_sub=None):
        self.n_sub = n_sub
    def fit(self, X, y=None):
        drop = []
        sub = [c for c in X.columns if c.startswith('sub__')]
        if self.n_sub is not None and sub:
            keep = set(X[sub].sum().sort_values(ascending=False).index[:self.n_sub])
            drop += [c for c in sub if c not in keep]
        tr = [c for c in X.columns if c.endswith('__trunc')]
        if tr:
            s = X[tr].sum()
            drop += list(s[s == 0].index)
        self.drop_ = drop
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

R_tr, R_te = ratio_feats(train), ratio_feats(test)
S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)
T_tr, T_te = F_tr['trunc'], F_te['trunc']
print('유전자 수:', len(genes), '| trunc 열:', T_tr.shape[1], '| train에서 1이 있는 trunc 열:', int((T_tr.sum() > 0).sum()))

Darwin | CPU arm64 | xgboost 3.4.1
유전자 수: 4384 | trunc 열: 4384 | train에서 1이 있는 trunc 열: 3664


#### trunc 열 train vs test (관찰용)
- 절단 변이가 있는 환자 비율이 test에서 크게 달라지는 유전자 (test 값은 피처 선택에 쓰지 않음)

In [2]:
share = pd.DataFrame({'train 환자 비율': T_tr.mean(), 'test 환자 비율': T_te.mean()})
share['test − train'] = share['test 환자 비율'] - share['train 환자 비율']
apc = T_tr['APC__trunc'] == 1
print(f"train APC 절단 변이 환자 {apc.sum()}명 중 COAD {(train.loc[apc, 'SUBCLASS'] == 'COAD').mean():.0%}")
share.sort_values('test − train', ascending=False).head(8).mul(100).round(1).astype(str) + '%'

train APC 절단 변이 환자 218명 중 COAD 71%


,train 환자 비율,test 환자 비율,test − train
APC__trunc,3.5%,24.2%,20.7%
SOX9__trunc,0.2%,4.0%,3.7%
TCF7L2__trunc,0.2%,3.8%,3.6%
ACVR2A__trunc,0.4%,3.8%,3.5%
TP53__trunc,8.4%,11.3%,2.9%
MYL1__trunc,0.0%,2.4%,2.4%
RPL22__trunc,0.6%,3.0%,2.4%
FBXW7__trunc,0.7%,2.9%,2.3%


#### 비교: 기준 / 23a, 22b / 23b
- 시간이 오래 걸림 (XGB 4개)

In [3]:
cat = lambda parts: pd.concat(parts, axis=1)
SETS = {
    '기준: any + stats':           (cat([F_tr['any'], F_tr['stats']]), cat([F_te['any'], F_te['stats']]), None),
    '23a: 기준 + trunc':           (cat([F_tr['any'], F_tr['stats'], T_tr]), cat([F_te['any'], F_te['stats'], T_te]), None),
    '22b: any + ratio + 치환 100':  (cat([F_tr['any'], R_tr, S_tr]), cat([F_te['any'], R_te, S_te]), 100),
    '23b: 22b + trunc':            (cat([F_tr['any'], R_tr, S_tr, T_tr]), cat([F_te['any'], R_te, S_te, T_te]), 100),
}
R = {}
for name, (X, X_te, n) in SETS.items():
    t = time.time()
    R[name] = run_cv(lambda: make_pipeline(FoldSelect(n), BalancedXGB(**BEST)), X, y, X_te, folds=folds, verbose=False)
    print(f"{name}: CV {R[name]['cv_mean']:.4f} ± {R[name]['cv_std']:.4f} (OOF {R[name]['oof_score']:.4f}, {time.time() - t:.0f}s)")

fs = lambda k: np.array(R[k]['fold_scores'])
PAIR = {'23a: 기준 + trunc': '기준: any + stats', '22b: any + ratio + 치환 100': '기준: any + stats',
        '23b: 22b + trunc': '22b: any + ratio + 치환 100'}
rows = []
for name in R:
    ref = PAIR.get(name)
    rows.append({'피처': name, 'CV': round(fs(name).mean(), 4), 'std': round(fs(name).std(), 4), 'OOF': round(R[name]['oof_score'], 4),
                 '비교 대상': ref or '-', '비교 대상 대비': round(fs(name).mean() - fs(ref).mean(), 4) if ref else '-',
                 '높은 fold': f'{int((fs(name) > fs(ref)).sum())}/5' if ref else '-'})
pd.DataFrame(rows)

기준: any + stats: CV 0.4125 ± 0.0109 (OOF 0.4131, 184s)
23a: 기준 + trunc: CV 0.4216 ± 0.0085 (OOF 0.4222, 303s)
22b: any + ratio + 치환 100: CV 0.4487 ± 0.0065 (OOF 0.4497, 196s)
23b: 22b + trunc: CV 0.4561 ± 0.0111 (OOF 0.4563, 306s)


,피처,CV,std,OOF,비교 대상,비교 대상 대비,높은 fold
0,기준: any + stats,0.4125,0.0109,0.4131,-,-,-
1,23a: 기준 + trunc,0.4216,0.0085,0.4222,기준: any + stats,0.0091,5/5
2,22b: any + ratio + 치환 100,0.4487,0.0065,0.4497,기준: any + stats,0.0362,5/5
3,23b: 22b + trunc,0.4561,0.0111,0.4563,22b: any + ratio + 치환 100,0.0074,4/5


#### fold별 점수

In [4]:
pd.DataFrame({k: np.round(fs(k), 4) for k in R}, index=[f'fold {i}' for i in range(5)])

,기준: any + stats,23a: 기준 + trunc,22b: any + ratio + 치환 100,23b: 22b + trunc
fold 0,0.4082,0.4119,0.4540,0.4432
fold 1,0.4230,0.4329,0.4426,0.4569
fold 2,0.4278,0.4298,0.4407,0.4476
fold 3,0.4023,0.4135,0.4485,0.4574
fold 4,0.4014,0.4199,0.4578,0.4755


#### 중요도 상위 trunc 열 (해석용)
- 23a를 train 전체로 한 번 학습해 어떤 trunc 열을 쓰는지 확인 (점수 계산용 아님)

In [5]:
X23a = SETS['23a: 기준 + trunc'][0]
pipe = make_pipeline(FoldSelect(None), BalancedXGB(**BEST)).fit(X23a, y)
cols = [c for c in X23a.columns if c not in pipe[0].drop_]
imp = pd.Series(pipe[-1].feature_importances_, index=cols)
rank = imp.rank(ascending=False).astype(int)
top = imp[imp.index.str.endswith('__trunc')].sort_values(ascending=False).head(10)
pd.DataFrame({'중요도': top.round(4), f'전체 {len(cols)}개 중 순위': rank[top.index]})

,중요도,전체 8057개 중 순위
TNFAIP3__trunc,0.0137,1
APC__trunc,0.0124,2
NPM1__trunc,0.0063,5
PTEN__trunc,0.0059,6
XYLT2__trunc,0.0050,8
SOCS1__trunc,0.0033,21
VHL__trunc,0.0030,24
ALB__trunc,0.0026,31
RREB1__trunc,0.0020,43
KLF3__trunc,0.0020,48


#### test 예측 분포 + APC 절단 변이 환자의 COAD 예측 (관찰용, 판단 기준 아님)
- 회의에서 걱정한 위험: trunc를 넣으면 COAD 예측이 더 늘어나는지
- train OOF에서 APC 절단 변이 환자의 COAD 예측 비율도 같이 봄 (train 실제 COAD 비율과 비교)

In [6]:
pred_te = {k: pd.Series(le.classes_[v['test_proba'].argmax(1)]) for k, v in R.items()}
dist = pd.DataFrame({k: p.value_counts(normalize=True) for k, p in pred_te.items()}).fillna(0)
print('test 예측 비율 (상위 8개)')
print((dist.sort_values('기준: any + stats', ascending=False).head(8) * 100).round(1).astype(str) + '%')

apc_te = (T_te['APC__trunc'] == 1).to_numpy()
apc_tr = (T_tr['APC__trunc'] == 1).to_numpy()
coad = list(le.classes_).index('COAD')
pd.DataFrame({
    'test COAD 예측': {k: f"{(p == 'COAD').mean():.1%}" for k, p in pred_te.items()},
    'test APC 절단 환자 중 COAD 예측': {k: f"{(p[apc_te] == 'COAD').mean():.1%}" for k, p in pred_te.items()},
    'train OOF APC 절단 환자 중 COAD 예측': {k: f"{(v['oof_proba'][apc_tr].argmax(1) == coad).mean():.1%}" for k, v in R.items()},
    'train OOF COAD 재현율': {k: f"{(v['oof_proba'][y == coad].argmax(1) == coad).mean():.1%}" for k, v in R.items()},
})

test 예측 비율 (상위 8개)
     기준: any + stats 23a: 기준 + trunc 22b: any + ratio + 치환 100 23b: 22b + trunc
STES           13.6%            9.8%                     21.8%            17.5%
COAD           10.9%           19.5%                     12.4%            17.5%
TGCT            5.5%            5.7%                      2.2%             2.3%
LUSC            5.1%            3.8%                      1.8%             1.7%
OV              4.9%            5.3%                      4.3%             4.7%
BRCA            4.9%            4.6%                      5.3%             4.6%
BLCA            4.4%            2.7%                      1.2%             0.9%
UCEC            3.9%            3.5%                      3.8%             3.7%


,test COAD 예측,test APC 절단 환자 중 COAD 예측,train OOF APC 절단 환자 중 COAD 예측,train OOF COAD 재현율
기준: any + stats,10.9%,41.2%,77.1%,68.2%
23a: 기준 + trunc,19.5%,79.6%,84.9%,68.2%
22b: any + ratio + 치환 100,12.4%,47.0%,77.1%,70.0%
23b: 22b + trunc,17.5%,70.3%,83.9%,71.3%


#### 23b + LR 섞기 vs C (리뷰 반영)
- 22 후보 C(22b XGB 0.7 + LR 0.3)가 **LB 0.3650**으로 새 기준 → 23b도 같은 방식(XGB 0.7 + LR 0.3)으로 섞어 C와 같은 fold로 짝 비교
- LR은 14·C와 같은 설정 (any, C=0.1, balanced), 비율 0.7 / 0.3도 같음 (다시 고르지 않음)
- 채택 기준: C보다 5 fold 중 4개 이상 상승 → 제출 후보 (C 시드 평균 다음)
- test COAD·STES 예측 비율도 같이 관찰 (판단 기준 아님)

In [7]:
from sklearn.linear_model import LogisticRegression
from src.validation import macro_f1

r_lr = run_cv(lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
              F_tr['any'].astype('float32'), y, F_te['any'].astype('float32'), folds=folds, verbose=False)
X22B, X23B = R['22b: any + ratio + 치환 100'], R['23b: 22b + trunc']
blend = lambda r: (0.7 * r['oof_proba'] + 0.3 * r_lr['oof_proba'], 0.7 * r['test_proba'] + 0.3 * r_lr['test_proba'])
BL = {'C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)': blend(X22B), '23b XGB 0.7 + LR 0.3': blend(X23B)}
fsb = lambda P: np.array([macro_f1(y[va], P[va].argmax(1)) for _, va in folds])
bC = fsb(BL['C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)'][0])
stes = list(le.classes_).index('STES')
rows = []
for name, (po, pt) in BL.items():
    s, pred = fsb(po), pt.argmax(1)
    rows.append({'조합': name, 'CV': round(s.mean(), 4), 'std': round(s.std(), 4), 'OOF': round(macro_f1(y, po.argmax(1)), 4),
                 'C 대비': round(s.mean() - bC.mean(), 4), 'C보다 높은 fold': f'{int((s > bC).sum())}/5',
                 'test COAD 예측': f'{np.mean(pred == coad):.1%}', 'test STES 예측': f'{np.mean(pred == stes):.1%}',
                 'test APC 절단 → COAD': f'{np.mean(pred[apc_te] == coad):.0%}'})
print('fold별:', {k: np.round(fsb(v[0]), 4).tolist() for k, v in BL.items()})
pd.DataFrame(rows)

fold별: {'C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)': [0.4594, 0.4437, 0.4564, 0.456, 0.4628], '23b XGB 0.7 + LR 0.3': [0.4474, 0.4608, 0.4584, 0.4597, 0.4867]}


,조합,CV,std,OOF,C 대비,C보다 높은 fold,test COAD 예측,test STES 예측,test APC 절단 → COAD
0,C: 22b XGB 0.7 + LR 0.3 (LB 0.3650),0.4557,0.0065,0.4565,0.000,0/5,15.0%,20.1%,57%
1,23b XGB 0.7 + LR 0.3,0.4626,0.0130,0.4627,0.007,4/5,19.5%,16.3%,77%


#### 시드 평균 5개 (팀 규칙: 실험 결과물 + 시드 평균 결과물)
- 17과 같은 방식: XGB만 시드 5개(42, 0, 1, 2, 3)로 학습해 확률 평균, fold(seed 42)와 LR·비율 0.7 / 0.3은 고정
- C와 23b + LR 모두 시드 평균판을 만들어 같은 fold로 비교 (시드 42판은 위 셀 결과를 그대로 사용)
- 시간이 오래 걸림 (XGB 8번 추가, 약 30분)

In [8]:
SEEDS = [42, 0, 1, 2, 3]

def xgb_seed_avg(name, n_sub):
    X, X_te, _ = SETS[name]
    oofs, tests = [R[name]['oof_proba']], [R[name]['test_proba']]   # 시드 42 = 위에서 돌린 결과
    for s in SEEDS[1:]:
        t = time.time()
        params = {**BEST, 'random_state': s}
        r = run_cv(lambda: make_pipeline(FoldSelect(n_sub), BalancedXGB(**params)), X, y, X_te, folds=folds, verbose=False)
        oofs.append(r['oof_proba']); tests.append(r['test_proba'])
        print(f"{name} 시드 {s}: CV {r['cv_mean']:.4f} ({time.time() - t:.0f}s)")
    return {'oof_proba': np.mean(oofs, axis=0), 'test_proba': np.mean(tests, axis=0)}

S22B, S23B = xgb_seed_avg('22b: any + ratio + 치환 100', 100), xgb_seed_avg('23b: 22b + trunc', 100)
BL_S = {'C 시드 평균': blend(S22B), '23b + LR 시드 평균': blend(S23B)}
bCs = fsb(BL_S['C 시드 평균'][0])
rows = []
for name, (po, pt) in {**BL, **BL_S}.items():
    s, pred = fsb(po), pt.argmax(1)
    rows.append({'조합': name, 'CV': round(s.mean(), 4), 'std': round(s.std(), 4), 'OOF': round(macro_f1(y, po.argmax(1)), 4),
                 'C(시드 42)보다 높은 fold': f'{int((s > bC).sum())}/5', 'C 시드 평균보다 높은 fold': f'{int((s > bCs).sum())}/5',
                 'test COAD 예측': f'{np.mean(pred == coad):.1%}', 'test STES 예측': f'{np.mean(pred == stes):.1%}'})
pd.DataFrame(rows)

22b: any + ratio + 치환 100 시드 0: CV 0.4507 (184s)
22b: any + ratio + 치환 100 시드 1: CV 0.4566 (184s)
22b: any + ratio + 치환 100 시드 2: CV 0.4534 (187s)
22b: any + ratio + 치환 100 시드 3: CV 0.4546 (180s)
23b: 22b + trunc 시드 0: CV 0.4579 (287s)
23b: 22b + trunc 시드 1: CV 0.4595 (284s)
23b: 22b + trunc 시드 2: CV 0.4559 (287s)
23b: 22b + trunc 시드 3: CV 0.4566 (283s)


,조합,CV,std,OOF,C(시드 42)보다 높은 fold,C 시드 평균보다 높은 fold,test COAD 예측,test STES 예측
0,C: 22b XGB 0.7 + LR 0.3 (LB 0.3650),0.4557,0.0065,0.4565,0/5,1/5,15.0%,20.1%
1,23b XGB 0.7 + LR 0.3,0.4626,0.0130,0.4627,4/5,2/5,19.5%,16.3%
2,C 시드 평균,0.4622,0.0082,0.4631,4/5,0/5,15.0%,20.0%
3,23b + LR 시드 평균,0.4688,0.0098,0.4690,4/5,4/5,19.0%,16.8%


In [9]:
# 결과물 2개 (시드 42판 + 시드 평균판), 데스크탑(Windows)에서만 저장 · 채택 여부는 C와의 fold 비교로 판단
OUTS = {'blend_trunc23b_0.7_LR0.3': (BL['23b XGB 0.7 + LR 0.3'], bC),
        'blend_trunc23b_0.7_LR0.3_seed5': (BL_S['23b + LR 시드 평균'], bCs)}
for fname, ((po, pt), ref) in OUTS.items():
    up = int((fsb(po) > ref).sum())
    print(f"{fname}: 비교 대상(C)보다 높은 fold {up}/5 → {'채택 기준 통과' if up >= 4 else '기준 미달'}")
    if platform.system() == 'Windows':
        save_submission(submission, pt, le, name=fname, cv_score=macro_f1(y, po.argmax(1)))   # 파일명 숫자 = OOF
if platform.system() != 'Windows':
    print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

blend_trunc23b_0.7_LR0.3: 비교 대상(C)보다 높은 fold 4/5 → 채택 기준 통과
blend_trunc23b_0.7_LR0.3_seed5: 비교 대상(C)보다 높은 fold 4/5 → 채택 기준 통과
Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


#### 제출 파일 (데스크탑에서만 저장)
- 23 설정 중 비교 대상보다 5 fold 중 4개 이상 오른 것 가운데 CV 최고를 저장

In [10]:
cands = {k: R[k] for k in R if k.startswith('23') and (fs(k) > fs(PAIR[k])).sum() >= 4}
if not cands:
    print('채택 기준(비교 대상보다 5 fold 중 4개 이상)을 넘는 23 설정 없음 → 저장 안 함')
else:
    best = max(cands, key=lambda k: cands[k]['cv_mean'])
    print('저장 대상:', best)
    if platform.system() == 'Windows':
        save_submission(submission, cands[best]['test_proba'], le, name=f'xgb_d3_cs0.3_trunc23{best[2]}', cv_score=cands[best]['cv_mean'])
    else:
        print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

저장 대상: 23b: 22b + trunc
Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


## 결과 (macOS M1)

**미리 확인 (관찰용)**: 유전자 4,384개 중 train에서 절단 변이가 한 번이라도 있는 유전자 3,664개. test에서 절단 변이 환자가 크게 늘어난 유전자는 APC(3.5% → 24.2%), SOX9, TCF7L2, ACVR2A 등 대장암 대표 유전자 (20과 같음). train APC 절단 변이 환자 218명 중 COAD 71%

| 피처 | CV (± std) | OOF | 비교 대상 | 대비 | 높은 fold |
| :--- | :-: | :-: | :--- | :-: | :-: |
| 기준: any + stats | 0.4125 ± 0.0109 | 0.4131 | - | - | - |
| **23a: 기준 + trunc** | 0.4216 ± 0.0085 | 0.4222 | 기준 | +0.0091 | **5/5** |
| 22b: any + ratio + 치환 100 | 0.4487 ± 0.0065 | 0.4497 | 기준 | +0.0362 | 5/5 |
| **23b: 22b + trunc** | **0.4561 ± 0.0111** | **0.4563** | 22b | +0.0074 | **4/5** (fold 0만 하락) |

- trunc를 더하면 기준(+0.009, 5/5)과 22b(+0.007, 4/5) 모두 오름 → **둘 다 채택 기준 통과**
  - 단, 23b의 상승폭(0.0074)은 std(0.0111)보다 작음 → CV로는 22b와 거의 같은 점수
- 중요도 상위 trunc 열: TNFAIP3(전체 1위), APC(2위), NPM1, PTEN, XYLT2, SOCS1, VHL → 암종 특이적인 종양억제유전자의 절단 변이를 사용

**test 예측 분포 (관찰용, 회의에서 걱정한 위험)**

| 피처 | test COAD 예측 | test APC 절단 환자 중 COAD 예측 | train OOF APC 절단 환자 중 COAD 예측 | test STES 예측 |
| :--- | :-: | :-: | :-: | :-: |
| 기준 | 10.9% | 41.2% | 77.1% | 13.6% |
| 23a: 기준 + trunc | **19.5%** | **79.6%** | 84.9% | 9.8% |
| 22b | 12.4% | 47.0% | 77.1% | 21.8% |
| 23b: 22b + trunc | **17.5%** | **70.3%** | 83.9% | 17.5% |

- **회의에서 걱정한 대로 trunc를 넣으면 test COAD 예측이 늘어남** (기준 10.9% → 23a 19.5%, 22b 12.4% → 23b 17.5%)
  - 늘어난 COAD는 대부분 APC 절단 변이 환자 (test APC 절단 환자 중 COAD 예측 41% → 80%)
  - train OOF에서도 APC 절단 환자를 실제 비율(71%)보다 많이 COAD로 예측 (77% → 85%) → trunc가 "APC 절단 = COAD" 규칙을 더 강하게 만듦
- 반대로 22b의 STES 쏠림은 줄어듦 (21.8% → 17.5%) → 22b에서 STES로 갔던 APC 절단 환자 일부가 COAD로 돌아옴
- 참고 (20의 대략적 계산): test APC 절단 환자 24.2% × train COAD 비율 71% ≈ 17%, LB 최고였던 14의 COAD 예측 17.3%
  - 23b(17.5%)는 이 값과 비슷, 23a(19.5%)는 더 많음 → 다만 대략적 계산이라 판단 기준으로 쓰지 않음 (test 분포로 튜닝하지 않음)

**23b + LR 섞기 vs C (리뷰 반영, 같은 fold 짝 비교)**

| 조합 | CV (± std) | OOF | C(시드 42)보다 높은 fold | C 시드 평균보다 높은 fold | test COAD | test STES |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: |
| C: 22b XGB 0.7 + LR 0.3 (LB 0.3650) | 0.4557 ± 0.0065 | 0.4565 | — | 1/5 | 15.0% | 20.1% |
| **23b XGB 0.7 + LR 0.3** | **0.4626 ± 0.0130** | 0.4627 | **4/5** (+0.0069) | 2/5 | 19.5% | 16.3% |
| C 시드 평균 (XGB 시드 5개) | 0.4622 ± 0.0082 | 0.4631 | 4/5 | — | 15.0% | 20.0% |
| **23b + LR 시드 평균** | **0.4688 ± 0.0098** | 0.4690 | 4/5 | **4/5** (+0.0066) | 19.0% | 16.8% |

- 23b + LR은 C보다 4/5 fold 높음 (fold 0만 하락: 0.4474 vs 0.4594) → **채택 기준 통과**
- 시드 평균끼리 비교해도 23b + LR 시드 평균이 C 시드 평균보다 4/5 fold 높음 (+0.0066)
- 단, 상승폭(0.007)이 std(0.010~0.013)보다 작음 → CV로는 차이가 크지 않음
- test APC 절단 환자 중 COAD 예측: C 57% → 23b + LR 77% → trunc의 COAD 쏠림은 섞어도 남음 (관찰용, LB 미확인)
- 시드 평균은 17과 같은 방식 (XGB만 시드 42, 0, 1, 2, 3, fold와 LR·비율 고정). 22b 시드별 CV 0.4487~0.4566, 23b 0.4559~0.4595

## 결론
- **H23a 채택 (CV 기준)**: 기준 + trunc가 기준 대비 +0.009, 5/5 fold
- **H23b 채택 (기준 통과)**: 22b + trunc가 CV 0.4561로 XGB 단독 최고 수준 (26c 0.4560과 같음), 다만 22b 대비 차이가 std 안
- ⚠️ **COAD 위험이 실제로 나타남**: trunc를 넣으면 test COAD 예측이 7~9%p 늘고, 대부분 APC 절단 변이 환자 → select6(LR_trunc 비중 최대, COAD 22.5%)이 LB 0.3183이었던 것을 생각하면 LB로 확인 필요
- **C 통과(LB 0.3650), 26c 기각** → 다음 후보는 23b
- **23b + LR 채택 기준 통과**: C보다 4/5 fold, 시드 평균끼리도 4/5 fold 상승 → C 시드 평균 다음 제출 후보
  - 23b는 22b의 STES 쏠림(20.1% → 16.3%)과 trunc의 COAD 쏠림(15.0% → 19.5%)이 서로 일부 상쇄되는 조합
- ⚠️ **STES 위험 (리뷰 반영)**: 22b 계열은 test STES 예측이 21.8%(train 6.1%, 기준 13.6%)로 크게 늘어 COAD보다 이쪽이 22 후보의 LB 위험일 수 있음 → 22 제출 결과(C LB 0.3650)와 함께 확인
- 제출 파일 (데스크탑에서 실행하면 저장)
  - `xgb_d3_cs0.3_trunc23b`: 23b XGB 단독 (채택 기준 통과 설정 중 CV 최고)
  - `blend_trunc23b_0.7_LR0.3`: 23b XGB 0.7 + LR 0.3 (시드 42)
  - `blend_trunc23b_0.7_LR0.3_seed5`: 같은 섞기의 XGB 시드 5개 평균판 (팀 규칙: 실험 결과물 + 시드 평균 결과물)
